# 🏨 Agente de Búsqueda de Hoteles con LangChain
## Agente ReAct · DuckDuckGo Search · Recomendación por ciudad y presupuesto

---

En esta actividad construirás un **agente de IA** que busca hoteles en internet y recomienda
la mejor opción precio-calidad dado una ciudad y un presupuesto.

```
Usuario
  │  ciudad: "Barcelona"  presupuesto: 80€/noche
  ▼
Interfaz Gradio
  │
  ▼
Agente ReAct (LangChain + Ollama)
  │
  ├── Razona: ¿qué buscar?
  ├── Actúa: busca en DuckDuckGo
  ├── Observa: analiza resultados
  └── Razona: ¿tengo suficiente info? → repite o responde
  │
  ▼
Recomendación: "Hotel X, €75/noche, valoración 8.5 — por estas razones..."
```

**A diferencia del RAG** (actividad 02), este agente decide de forma autónoma **cuántas búsquedas hacer
y qué buscar** para llegar a la respuesta más completa.

**⏱ Duración:** 90 minutos | **🎯 Resultado:** App web con agente de recomendación hotelera

---
## PARTE 1 · Instalación del entorno
**⏱ 3 minutos**

In [ ]:
# Instalar todas las dependencias en una sola celda
# langchain-community incluye DuckDuckGoSearchRun y otras herramientas
# duckduckgo-search es el backend que usa LangChain para las búsquedas
!pip install -q \
    langchain \
    langchain-community \
    langchain-ollama \
    duckduckgo-search \
    gradio \
    pyngrok
print('✅ Dependencias instaladas')

---
## PARTE 2 · Configurar Ollama y el modelo
**⏱ 5 minutos**

El agente necesita un modelo que sea capaz de **razonar en pasos** y **decidir qué herramienta usar**.
Para esto usamos `llama3.2:1b` — más capaz que TinyLlama para seguir instrucciones complejas.

> 💡 **Alternativa:** Si ya tienes el servidor Ollama corriendo desde la **Actividad 00a**,
> puedes saltarte la instalación y apuntar el modelo a esa URL pública.

In [ ]:
# Instalar Ollama en el sistema operativo de Colab
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama instalado')

In [ ]:
import subprocess
import time
import requests

# Lanzar el servidor Ollama en background
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Esperar hasta que el servidor esté listo
print('Arrancando Ollama...', end='')
for _ in range(30):
    try:
        if requests.get('http://localhost:11434', timeout=2).status_code == 200:
            print(' listo')
            break
    except:
        print('.', end='', flush=True)
        time.sleep(1)

print('✅ Servidor Ollama activo en localhost:11434')

In [ ]:
# 🔧 PARÁMETRO: modelo a usar para el agente
# llama3.2:1b es mejor que tinyllama para razonamiento y seguimiento de instrucciones
# Alternativas: qwen2.5:1.5b (~1GB), llama3.2:3b (~2GB, más capaz pero más lento)
MODELO = 'llama3.2:1b'

print(f'Descargando {MODELO}... (puede tardar 2-4 min)')
resultado = subprocess.run(['ollama', 'pull', MODELO], capture_output=True, text=True)

if resultado.returncode == 0:
    print(f'✅ Modelo {MODELO} listo')
else:
    print(f'❌ Error: {resultado.stderr}')

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Instanciar el modelo — este objeto lo usaremos en todo el notebook
llm = ChatOllama(
    model=MODELO,
    temperature=0.1,    # Baja temperatura = respuestas más consistentes y menos creativas
                        # Útil para agentes que necesitan seguir un formato estricto
)

# Prueba rápida para confirmar que el modelo responde
test = llm.invoke([HumanMessage(content='¿Cuál es la capital de España? Solo el nombre.')])
print(f'Respuesta de prueba: {test.content}')
print('✅ Modelo listo para usar')

---
## PARTE 3 · Concepto: ¿Qué es un agente ReAct?

Un **agente** es un LLM que puede usar **herramientas** para obtener información del mundo real
y actuar en función de ella. El patrón **ReAct** (Reason + Act) funciona así:

```
RAZONA  → "Necesito buscar hoteles en Barcelona con precio < 80€"
ACTÚA   → llama a DuckDuckGo con "hoteles Barcelona menos de 80 euros noche"
OBSERVA → lee los resultados de búsqueda
RAZONA  → "Los resultados mencionan 3 hoteles, necesito más info sobre el mejor"
ACTÚA   → llama a DuckDuckGo con "Hotel X Barcelona reseñas precio 2024"
OBSERVA → lee las reseñas
RAZONA  → "Tengo suficiente información para responder"
RESPONDE → "Mi recomendación es Hotel X por..."
```

La diferencia clave con RAG es que el agente **decide autónomamente** cuántas búsquedas hacer
y qué buscar en cada paso. No sigue un pipeline fijo.

### Herramienta: DuckDuckGo Search
DuckDuckGo ofrece búsqueda web **gratuita y sin API key**. LangChain incluye un wrapper
que convierte cualquier búsqueda en texto estructurado que el agente puede leer.

---
## PARTE 4 · Configurar y probar DuckDuckGo
**⏱ 5 minutos**

Antes de integrar DuckDuckGo en el agente, lo probamos de forma independiente
para ver qué tipo de resultados devuelve.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# Configurar el wrapper con límite de resultados
# max_results controla cuántos snippets devuelve por búsqueda
ddg_wrapper = DuckDuckGoSearchAPIWrapper(
    max_results=5,           # 5 resultados por búsqueda es suficiente para el agente
    region='es-es',          # Preferir resultados en español
    time='y'                 # Resultados del último año (hoteles actualizados)
)

# DuckDuckGoSearchRun convierte el wrapper en una herramienta que LangChain puede invocar
herramienta_busqueda = DuckDuckGoSearchRun(
    api_wrapper=ddg_wrapper,
    name='busqueda_web',                          # Nombre que el agente usará para invocarla
    description=(
        'Busca información actualizada en internet sobre hoteles, precios y valoraciones. '
        'Útil para encontrar opciones de alojamiento en cualquier ciudad con presupuesto específico. '
        'Input: cadena de texto con la consulta de búsqueda.'
    )
)

print('✅ Herramienta DuckDuckGo configurada')
print(f'   Nombre     : {herramienta_busqueda.name}')
print(f'   Descripción: {herramienta_busqueda.description[:80]}...')

In [ ]:
# Probar la herramienta con una búsqueda de hotel real
# Así vemos el formato de los resultados ANTES de que el agente los procese
print('=== TEST: búsqueda directa con DuckDuckGo ===')
print('Consulta: "hoteles Barcelona menos de 80 euros noche"')
print()

resultado_ddg = herramienta_busqueda.invoke('hoteles Barcelona menos de 80 euros por noche valoracion')

# Mostrar los primeros 800 caracteres del resultado
print(resultado_ddg[:800])
print('...')
print(f'\nTotal de caracteres recibidos: {len(resultado_ddg)}')
print('✅ DuckDuckGo devuelve resultados correctamente')

---
## PARTE 5 · Diseñar el prompt del agente
**⏱ 5 minutos**

El **system prompt** es la instrucción que define el comportamiento del agente.
Un buen prompt de agente debe:
1. Definir el rol y el objetivo
2. Indicar qué herramientas tiene disponibles
3. Especificar el **formato de razonamiento** (ReAct: Thought → Action → Observation)
4. Definir el formato de la respuesta final

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt del sistema: define quién es el agente y cómo debe comportarse
SYSTEM_PROMPT = """\
Eres un experto en viajes y reservas de hoteles. Tu misión es ayudar al usuario a encontrar
el mejor hotel precio-calidad en una ciudad específica, respetando su presupuesto máximo.

PROCESO A SEGUIR:
1. Busca hoteles en la ciudad indicada con el presupuesto dado
2. Si los primeros resultados no son suficientes, haz búsquedas adicionales más específicas
3. Compara opciones considerando: precio por noche, valoración/puntuación, ubicación y servicios
4. Da una recomendación clara con justificación

FORMATO DE RESPUESTA FINAL:
Cuando tengas suficiente información, responde con esta estructura:
🏨 **Recomendación principal:** [nombre del hotel]
💰 **Precio estimado:** [precio por noche]
⭐ **Valoración:** [puntuación si la encontraste]
📍 **Ubicación:** [zona o barrio]
✅ **Por qué es la mejor opción:** [2-3 razones concretas]
🔍 **Alternativa:** [otro hotel si el usuario quiere comparar]

Responde siempre en español.
"""

print('=== SYSTEM PROMPT DEL AGENTE ===')
print(SYSTEM_PROMPT)
print('✅ Prompt definido')

---
## PARTE 6 · Construir el agente ReAct
**⏱ 8 minutos**

Usamos `initialize_agent` de LangChain con el tipo `ZERO_SHOT_REACT_DESCRIPTION`.
Este tipo de agente:
- No necesita ejemplos previos (zero-shot)
- Usa el patrón ReAct automáticamente
- Decide qué herramienta usar basándose en la descripción de cada herramienta

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain_core.messages import SystemMessage

# Lista de herramientas disponibles para el agente
# El agente puede usar cualquiera de estas según lo que necesite
tools = [herramienta_busqueda]

# Crear el agente combinando LLM + herramientas + tipo de razonamiento
agente = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Patrón ReAct sin ejemplos previos
    verbose=True,              # verbose=True muestra el razonamiento paso a paso
    handle_parsing_errors=True, # Si el modelo genera formato incorrecto, reintenta
    max_iterations=6,          # Máximo de pasos Thought→Action→Observation
                               # Evita bucles infinitos si el modelo se pierde
    agent_kwargs={
        'system_message': SystemMessage(content=SYSTEM_PROMPT)
    }
)

print('✅ Agente creado')
print(f'   Herramientas  : {[t.name for t in tools]}')
print(f'   Tipo de agente: ZERO_SHOT_REACT_DESCRIPTION')
print(f'   Iteraciones   : máximo 6')

In [ ]:
# Probar el agente con una consulta real
# verbose=True en el agente mostrará cada paso del razonamiento
print('=== PRUEBA DEL AGENTE ===')
print('Pregunta: Hotel en Madrid con presupuesto de 70€/noche')
print('-' * 50)

respuesta_test = agente.invoke({
    'input': 'Busca el mejor hotel en Madrid con un presupuesto de 70 euros por noche. '
             'Quiero buena valoración y buena ubicación.'
})

print('\n=== RESPUESTA FINAL ===')
print(respuesta_test['output'])
print('\n✅ Agente funciona correctamente')

### 💬 Reflexión 1
> **Observa el razonamiento (Thought/Action/Observation) que mostró el agente.**
> ¿Cuántas búsquedas hizo? ¿Modificó la consulta entre búsquedas?
> ¿En qué momento decidió que tenía suficiente información para responder?
>
> *(Escribe tu respuesta aquí haciendo doble clic en esta celda)*

---

---
## PARTE 7 · Función de consulta con formato de entrada estructurado
**⏱ 5 minutos**

Creamos una función que recibe los parámetros por separado (ciudad, presupuesto, preferencias)
y construye la consulta óptima para el agente.

In [ ]:
def buscar_hotel(ciudad: str, presupuesto: int, preferencias: str = '') -> str:
    """
    Busca el mejor hotel para los parámetros dados usando el agente de IA.
    
    Args:
        ciudad: nombre de la ciudad donde buscar (ej: 'Barcelona')
        presupuesto: precio máximo por noche en euros
        preferencias: requisitos adicionales (ej: 'piscina', 'cerca del centro', 'desayuno')
    
    Returns:
        Recomendación del agente como texto formateado
    """
    if not ciudad.strip():
        return '❌ Por favor, indica una ciudad.'

    if presupuesto <= 0:
        return '❌ El presupuesto debe ser mayor que 0.'

    # Construir la consulta combinando todos los parámetros
    consulta = (
        f'Busca el mejor hotel en {ciudad} con un presupuesto máximo de {presupuesto} euros por noche. '
        f'Quiero la mejor relación precio-calidad con buena valoración de los huéspedes.'
    )

    # Añadir preferencias adicionales si el usuario las indicó
    if preferencias.strip():
        consulta += f' Preferencias adicionales: {preferencias}.'

    try:
        resultado = agente.invoke({'input': consulta})
        return resultado['output']
    except Exception as e:
        return f'❌ Error al consultar el agente: {str(e)}'


print('✅ Función buscar_hotel definida')

In [ ]:
# Probar la función con distintos escenarios
print('=== PRUEBA 1: Sevilla, presupuesto bajo ===')
resultado_1 = buscar_hotel('Sevilla', 60, 'céntrico, buenas reseñas')
print(resultado_1)
print()
print('✅ Función de búsqueda probada')

---
## PARTE 8 · Interfaz Gradio para el agente
**⏱ 10 minutos**

Construimos una interfaz web donde el usuario puede:
- Introducir la ciudad y el presupuesto
- Añadir preferencias adicionales
- Ver la recomendación del agente en formato rico
- Ver el historial de búsquedas de la sesión

In [ ]:
import gradio as gr

# Historial de búsquedas de la sesión
historial_sesion = []


def buscar_con_historial(ciudad, presupuesto, preferencias):
    """Wrapper que actualiza el historial de búsquedas."""
    if not ciudad.strip():
        return '❌ Por favor, introduce una ciudad.', _formatear_historial()

    # Llamar al agente
    resultado = buscar_hotel(ciudad.strip(), int(presupuesto), preferencias.strip())

    # Guardar en historial
    historial_sesion.append({
        'ciudad': ciudad.strip(),
        'presupuesto': int(presupuesto),
        'resultado_resumen': resultado[:100] + '...' if len(resultado) > 100 else resultado
    })

    return resultado, _formatear_historial()


def _formatear_historial():
    """Formatea el historial como texto markdown."""
    if not historial_sesion:
        return '*Sin búsquedas todavía.*'
    lineas = []
    for i, h in enumerate(reversed(historial_sesion[-5:]), 1):  # Últimas 5
        lineas.append(f'**{i}.** {h["ciudad"]} — €{h["presupuesto"]}/noche')
    return '\n'.join(lineas)


# CSS personalizado para la interfaz
CSS = """
    .gradio-container { font-family: 'Segoe UI', sans-serif; }
    .output-box { background: #f8f9fa; border-radius: 8px; padding: 15px; }
    h1 { color: #1a1a2e; }
"""

# Construir la interfaz con gr.Blocks para control total del layout
with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title='🏨 Buscador de Hoteles IA') as demo:

    gr.Markdown(
        '# 🏨 Buscador de Hoteles con IA\n'
        'Introduce tu destino y presupuesto. El agente buscará en internet la mejor opción para ti.'
    )

    with gr.Row():
        # Panel izquierdo: parámetros de búsqueda
        with gr.Column(scale=1):
            gr.Markdown('### 🔍 Parámetros de búsqueda')

            ciudad_input = gr.Textbox(
                label='Ciudad de destino',
                placeholder='Ej: Barcelona, Madrid, Roma, París...',
                max_lines=1
            )

            presupuesto_input = gr.Slider(
                label='Presupuesto máximo por noche (€)',
                minimum=20,
                maximum=500,
                value=80,         # Valor por defecto: 80€
                step=10,
                info='Precio máximo que estás dispuesto a pagar por noche'
            )

            preferencias_input = gr.Textbox(
                label='Preferencias adicionales (opcional)',
                placeholder='Ej: piscina, desayuno incluido, cerca del centro, parking...',
                max_lines=2
            )

            btn_buscar = gr.Button('🔎 Buscar mejor hotel', variant='primary', size='lg')
            btn_limpiar = gr.ClearButton(
                [ciudad_input, preferencias_input],
                value='🗑️ Limpiar'
            )

            gr.Markdown('---')
            gr.Markdown('### 🕐 Búsquedas recientes')
            historial_output = gr.Markdown('*Sin búsquedas todavía.*')

        # Panel derecho: resultado del agente
        with gr.Column(scale=2):
            gr.Markdown('### 🏨 Recomendación del agente')
            resultado_output = gr.Markdown(
                value='*Aquí aparecerá la recomendación del agente...*',
                elem_classes=['output-box']
            )

    # Ejemplos predefinidos para que el estudiante pruebe sin escribir
    gr.Examples(
        examples=[
            ['Barcelona',  80,  'cerca de la Sagrada Familia'],
            ['Madrid',     60,  'zona Gran Vía, desayuno incluido'],
            ['Lisboa',     70,  ''],
            ['Roma',       90,  'cerca del Coliseo'],
            ['Sevilla',    50,  'económico, buenas reseñas'],
            ['París',     120,  'romántico, vistas'],
        ],
        inputs=[ciudad_input, presupuesto_input, preferencias_input],
        label='Ejemplos de búsqueda'
    )

    # Conectar el botón con la función de búsqueda
    btn_buscar.click(
        fn=buscar_con_historial,
        inputs=[ciudad_input, presupuesto_input, preferencias_input],
        outputs=[resultado_output, historial_output]
    )

    # También permitir búsqueda al presionar Enter en el campo de ciudad
    ciudad_input.submit(
        fn=buscar_con_historial,
        inputs=[ciudad_input, presupuesto_input, preferencias_input],
        outputs=[resultado_output, historial_output]
    )

print('✅ Interfaz Gradio construida')

In [ ]:
# Lanzar la interfaz en modo local para probarla antes de exponer con Ngrok
demo.launch(
    server_name='0.0.0.0',    # Necesario para que Ngrok pueda conectarse
    server_port=7860,          # Puerto estándar de Gradio
    share=False,               # No usar el share de Gradio, usaremos Ngrok
    quiet=True
)
print('✅ Interfaz corriendo en localhost:7860')

### 💬 Reflexión 2
> 1. **¿En qué se diferencia este agente del sistema RAG de la Actividad 02?**
>    Pista: piensa en quién decide qué buscar y cuántas veces.
> 2. **¿Por qué usamos `temperature=0.1` en lugar de 0.7 u 0.9 para el agente?**
>    ¿Qué pasaría si subiéramos la temperatura?
>
> *(Escribe tu respuesta aquí)*

---

---
## PARTE 9 · Reflexión final

> **1. Prueba el agente con una ciudad pequeña o poco turística (ej: Cuenca, Évora, Matera).**
>    ¿Encuentra hoteles? ¿La calidad de la recomendación cambia? ¿Por qué?
>
> *(Escribe aquí)*

> **2. ¿Qué limitaciones tiene este agente para un uso real en una app de viajes?**
>    Piensa en: precios en tiempo real, disponibilidad, reserva directa, idioma.
>
> *(Escribe aquí)*

> **3. El agente puede hacer hasta 6 búsquedas (`max_iterations=6`). ¿Qué ventajas y riesgos tiene aumentar ese número?**
>
> *(Escribe aquí)*

> **4. ¿Cómo añadirías una herramienta de conversión de divisas para que el agente pueda buscar también en libras o dólares?**
>    (No tienes que implementarlo, solo describe el diseño.)
>
> *(Escribe aquí)*

---

---
## PARTE 10 · Retos opcionales

**Reto A — Segunda herramienta: conversión de moneda**
Crea una segunda herramienta que el agente pueda usar para convertir el presupuesto
del usuario a la moneda local del destino (útil para hoteles fuera de la zona euro).
Pista: usa `requests` para llamar a una API de divisas gratuita como `frankfurter.app`.

**Reto B — Memoria de conversación**
Modifica el agente para que recuerde las búsquedas anteriores dentro de la sesión.
Así el usuario puede decir «muéstrame algo más barato» y el agente entiende el contexto.
Pista: añade `ConversationBufferMemory` al agente (ver Actividad 05).

**Reto C — Dominio personalizado Ngrok**
Si tienes un dominio estático en Ngrok (plan gratuito permite 1),
configúralo en la celda de Ngrok para que la URL sea siempre la misma.

In [ ]:
# 🔧 Espacio para tus experimentos


---
## PARTE 11 · Publicación con Ngrok
**⏱ 5 minutos**

Exponemos la interfaz al exterior para compartirla con la clase.

In [ ]:
# pyngrok ya está instalado desde PARTE 1
# Verificamos que está disponible
from pyngrok import ngrok
print('✅ pyngrok disponible')

In [ ]:
from google.colab import userdata

# Leer token desde Colab Secrets (más seguro que pegarlo en el código)
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    print('✅ Token leído desde Colab Secrets')
except Exception:
    NGROK_TOKEN = 'PEGA_TU_TOKEN_AQUÍ'   # Solo si no tienes Secrets configurado
    print('⚠️  Usando token manual. Configura Colab Secrets para mayor seguridad.')

ngrok.set_auth_token(NGROK_TOKEN)
print('✅ Ngrok autenticado')

In [ ]:
# Cerrar interfaz y túnel previos para evitar conflictos
ngrok.kill()
demo.close()

# Crear túnel apuntando al puerto de Gradio
tunnel = ngrok.connect(7860, 'http')
URL_PUBLICA = tunnel.public_url

# Relanzar la interfaz con la URL del túnel activo
demo.launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=False,
    quiet=True
)

print('=' * 60)
print('🌐 AGENTE DE HOTELES DISPONIBLE EN:')
print(f'   {URL_PUBLICA}')
print('=' * 60)
print('Comparte esta URL con la clase para que prueben el agente.')

In [ ]:
# Verificar que el túnel está activo
tunnels_activos = ngrok.get_tunnels()
print('=== TÚNELES ACTIVOS ===')
for t in tunnels_activos:
    print(f'  URL pública : {t.public_url}')
    print(f'  Puerto local: {t.config["addr"]}')
print('✅ Túnel verificado')

---
## 📊 Rúbrica de evaluación

| Criterio | Excelente (5) | Satisfactorio (3) | En desarrollo (1) |
|---|---|---|---|
| **Ejecución del notebook** | Todas las celdas sin errores de arriba a abajo | Algún error menor no bloqueante | Más de 2 partes no ejecutan |
| **Agente funcional** | El agente busca y recomienda hoteles con justificación clara | Responde pero sin estructura completa | No genera recomendaciones útiles |
| **Interfaz Gradio** | Interfaz cargada, ejemplos funcionando, historial visible | La interfaz carga pero con errores parciales | La interfaz no carga |
| **Publicación con Ngrok** | URL pública generada y compartida con el docente | URL generada pero acceso intermitente | No se pudo generar la URL |
| **Reflexión escrita** | Las 4 preguntas respondidas con detalle y ejemplos propios | Al menos 2 preguntas respondidas correctamente | Sin respuestas o respuestas superficiales |
| **Reto opcional** | Implementó y documentó al menos un reto | Intentó un reto con código parcial | No intentó los retos |

---
## 🧹 Limpieza final (ejecutar al terminar)

In [ ]:
# Liberar recursos en orden
ngrok.kill()                    # 1. Cerrar el túnel Ngrok
demo.close()                    # 2. Cerrar la interfaz Gradio
ollama_process.terminate()      # 3. Detener el servidor Ollama
print('✅ Todos los recursos liberados correctamente')